# 07. 근거 설명 및 센서 중요도

이 노트북은 05/06 모델 산출물을 읽어 Priority Engine과 Agent가 사용할 수 있는 decision feature 근거를 만든다.

- 위험도/리드타임 모델의 feature importance 정리
- 고위험 window별 top feature 근거 생성
- group별 과대위험 판단 진단 정리
- 08 decision_features/export에서 사용할 evidence table 생성

이번 07은 모델을 새로 학습하지 않는다. 06 결과를 우선순위 회귀/스코어링 입력으로 쓰기 좋은 형태로 변환하는 단계다.

In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)

RUN_ID = datetime.now().strftime("run_%Y%m%d_%H%M%S")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    candidates = [PROJECT_ROOT, *PROJECT_ROOT.parents]
    PROJECT_ROOT = next(path for path in candidates if (path / "data").exists())

FEATURE_DIR = PROJECT_ROOT / "data" / "processed" / "ml_features"
SUPERVISED_DIR = PROJECT_ROOT / "data" / "processed" / "ml_supervised"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "ml_explainability"
RUN_DIR = OUTPUT_DIR / "runs" / RUN_ID
PLOT_DIR = RUN_DIR / "plots"

for path in [OUTPUT_DIR, RUN_DIR, PLOT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

TRAINABLE_PATH = FEATURE_DIR / "trainable_windows.csv"
FEATURE_COLUMNS_PATH = FEATURE_DIR / "feature_columns.csv"
AGENT_OUTPUTS_PATH = SUPERVISED_DIR / "agent_model_outputs.csv"
IMPORTANCE_PATH = SUPERVISED_DIR / "risk_leadtime_feature_importance.csv"
GROUP_DIAGNOSTICS_PATH = SUPERVISED_DIR / "risk_group_diagnostics.csv"
THRESHOLD_DIAGNOSTICS_PATH = SUPERVISED_DIR / "risk_threshold_diagnostics.csv"
METADATA_PATH = SUPERVISED_DIR / "risk_leadtime_model_metadata.json"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUN_ID:", RUN_ID)

## 1. 입력 데이터 로드

07은 06의 최신 산출물을 입력으로 사용한다. `data/processed/` 아래 산출물은 Git 추적 대상이 아니므로, 06 실행 후 07을 실행해야 한다.

In [ ]:
required_paths = [
    TRAINABLE_PATH,
    FEATURE_COLUMNS_PATH,
    AGENT_OUTPUTS_PATH,
    IMPORTANCE_PATH,
    GROUP_DIAGNOSTICS_PATH,
    THRESHOLD_DIAGNOSTICS_PATH,
    METADATA_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("07 실행 전에 06 산출물이 필요합니다: " + ", ".join(missing_paths))

windows = pd.read_csv(TRAINABLE_PATH)
feature_columns = pd.read_csv(FEATURE_COLUMNS_PATH)
agent_outputs = pd.read_csv(AGENT_OUTPUTS_PATH)
feature_importance = pd.read_csv(IMPORTANCE_PATH)
group_diagnostics = pd.read_csv(GROUP_DIAGNOSTICS_PATH)
threshold_diagnostics = pd.read_csv(THRESHOLD_DIAGNOSTICS_PATH)
metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))

merge_keys = ["manufacturer", "substation_id", "source_file", "window_start", "window_end"]
df = agent_outputs.merge(windows, on=merge_keys, how="left", suffixes=("", "__feature"), validate="one_to_one")

print("agent_outputs:", agent_outputs.shape)
print("feature_importance:", feature_importance.shape)
print("group_diagnostics:", group_diagnostics.shape)
print("merged:", df.shape)
display(agent_outputs.head())

## 2. Feature importance 요약

LightGBM의 기본 feature importance를 모델별로 정리한다. 이는 인과 설명이 아니라 모델이 분기에서 자주 사용한 feature의 중요도다.

In [ ]:
feature_name_col = "column" if "column" in feature_columns.columns else "feature"
family_map = feature_columns.set_index(feature_name_col)["family"].to_dict() if "family" in feature_columns.columns else {}

importance_summary = feature_importance.copy()
importance_summary["family"] = importance_summary["feature"].map(family_map).fillna("모델결과/기타")
importance_summary["importance_rank"] = importance_summary.groupby("model")["importance"].rank(method="first", ascending=False).astype(int)
importance_summary["importance_share"] = importance_summary["importance"] / importance_summary.groupby("model")["importance"].transform("sum")

top_importance = importance_summary.sort_values(["model", "importance_rank"]).groupby("model").head(30).reset_index(drop=True)
family_importance = importance_summary.groupby(["model", "family"], as_index=False)["importance"].sum()
family_importance["importance_share"] = family_importance["importance"] / family_importance.groupby("model")["importance"].transform("sum")
family_importance = family_importance.sort_values(["model", "importance"], ascending=[True, False]).reset_index(drop=True)

display(top_importance.head(20))
display(family_importance)

In [ ]:
for model_name, model_frame in top_importance.groupby("model"):
    plot_frame = model_frame.head(15).sort_values("importance")
    plt.figure(figsize=(9, 6))
    plt.barh(plot_frame["feature"], plot_frame["importance"])
    plt.title(f"Top feature importance - {model_name}")
    plt.xlabel("LightGBM importance")
    plt.tight_layout()
    latest_path = OUTPUT_DIR / f"{model_name}_top_feature_importance.png"
    run_path = PLOT_DIR / f"{model_name}_top_feature_importance.png"
    plt.savefig(latest_path, dpi=150, bbox_inches="tight")
    plt.savefig(run_path, dpi=150, bbox_inches="tight")
    plt.show()

## 3. Window별 근거 feature 생성

각 window에서 모델 중요도가 높은 feature 중 해당 row의 값이 전체 분포에서 상대적으로 두드러지는 feature를 top evidence로 잡는다. SHAP이 아니므로 방향성 있는 국소 기여도는 아니지만, Priority Engine과 Agent가 우선순위 판단 근거로 사용할 수 있는 baseline 근거가 된다.

In [ ]:
risk_top_features = top_importance[top_importance["model"].eq("risk_lgbm")]["feature"].head(25).tolist()
leadtime_top_features = top_importance[top_importance["model"].eq("leadtime_lgbm")]["feature"].head(25).tolist()
evidence_features = [feature for feature in dict.fromkeys(risk_top_features + leadtime_top_features) if feature in df.columns]

reference_mask = df["split_time_based"].eq("train") & df["label"].eq("normal")
reference = df.loc[reference_mask, evidence_features].apply(pd.to_numeric, errors="coerce")
feature_median = reference.median(numeric_only=True)
feature_iqr = reference.quantile(0.75) - reference.quantile(0.25)
feature_iqr = feature_iqr.replace(0, np.nan)

importance_lookup = importance_summary.groupby("feature")["importance_share"].max().to_dict()

def top_evidence_for_row(row: pd.Series, top_n: int = 5) -> list[dict]:
    items = []
    for feature in evidence_features:
        value = pd.to_numeric(row.get(feature), errors="coerce")
        if pd.isna(value):
            continue
        median = feature_median.get(feature, np.nan)
        iqr = feature_iqr.get(feature, np.nan)
        if pd.isna(median) or pd.isna(iqr) or iqr == 0:
            continue
        robust_z = (value - median) / iqr
        score = abs(robust_z) * float(importance_lookup.get(feature, 0.0))
        direction = "high" if robust_z > 0 else "low"
        items.append({
            "feature": feature,
            "value": float(value),
            "reference_median": float(median),
            "robust_z": float(robust_z),
            "direction": direction,
            "evidence_score": float(score),
            "family": family_map.get(feature, "모델결과/기타"),
        })
    return sorted(items, key=lambda item: item["evidence_score"], reverse=True)[:top_n]

explain_mask = df["risk_level"].isin(["high", "medium"]) | df["anomaly_label"].eq(1)
explain_candidates = df.loc[explain_mask].copy()
explain_candidates = explain_candidates.sort_values(["risk_probability", "anomaly_score"], ascending=[False, False]).head(500)

evidence_rows = []
for _, row in explain_candidates.iterrows():
    evidences = top_evidence_for_row(row, top_n=5)
    top_sensors = [item["feature"] for item in evidences]
    sensor_scores = {item["feature"]: item["evidence_score"] for item in evidences}
    pattern_notes = []
    if row.get("risk_class", 0) == 1:
        pattern_notes.append("risk threshold를 넘은 고위험 window")
    elif row.get("risk_level") == "medium":
        pattern_notes.append("risk probability가 중간 이상인 관찰 후보")
    if row.get("anomaly_label", 0) == 1:
        pattern_notes.append("Isolation Forest anomaly label이 1")
    if row.get("lead_time_bucket"):
        pattern_notes.append(f"leadtime 후보: {row.get('lead_time_bucket')}")

    evidence_rows.append({
        "manufacturer": row.get("manufacturer"),
        "substation_id": row.get("substation_id"),
        "source_file": row.get("source_file"),
        "window_start": row.get("window_start"),
        "window_end": row.get("window_end"),
        "label": row.get("label"),
        "split_time_based": row.get("split_time_based"),
        "configuration_type": row.get("configuration_type"),
        "season_bucket": row.get("season_bucket"),
        "anomaly_score": row.get("anomaly_score"),
        "anomaly_label": row.get("anomaly_label"),
        "risk_probability": row.get("risk_probability"),
        "risk_class": row.get("risk_class"),
        "risk_level": row.get("risk_level"),
        "lead_time_bucket": row.get("lead_time_bucket"),
        "lead_time_confidence": row.get("lead_time_confidence"),
        "top_sensors": json.dumps(top_sensors, ensure_ascii=False),
        "sensor_scores": json.dumps(sensor_scores, ensure_ascii=False),
        "evidence_details": json.dumps(evidences, ensure_ascii=False),
        "pattern_notes": " | ".join(pattern_notes),
        "explainability_run_id": RUN_ID,
        "source_model_run_id": metadata.get("run_id"),
    })

window_evidence = pd.DataFrame(evidence_rows)
display(window_evidence.head(20))

## 4. Holdout 과대위험 window 전용 근거

우선순위 모델 관점에서 중요한 오류는 정상 window가 지나치게 높은 위험 입력값을 받는 경우다. 따라서 holdout 정상 window 중 risk threshold를 넘은 과대위험 사례를 따로 분리해 evidence를 저장한다.

In [ ]:
fp_mask = (
    df["split_time_based"].eq("holdout")
    & df["label"].eq("normal")
    & df["risk_class"].eq(1)
)
false_positive_windows = df.loc[fp_mask].copy()
false_positive_windows = false_positive_windows.sort_values("risk_probability", ascending=False)

fp_rows = []
for _, row in false_positive_windows.iterrows():
    evidences = top_evidence_for_row(row, top_n=8)
    top_sensors = [item["feature"] for item in evidences]
    sensor_scores = {item["feature"]: item["evidence_score"] for item in evidences}
    fp_rows.append({
        "manufacturer": row.get("manufacturer"),
        "substation_id": row.get("substation_id"),
        "source_file": row.get("source_file"),
        "window_start": row.get("window_start"),
        "window_end": row.get("window_end"),
        "label": row.get("label"),
        "split_time_based": row.get("split_time_based"),
        "split_regime_based": row.get("split_regime_based"),
        "manufacturer": row.get("manufacturer"),
        "configuration_type": row.get("configuration_type"),
        "season_bucket": row.get("season_bucket"),
        "anomaly_score": row.get("anomaly_score"),
        "anomaly_label": row.get("anomaly_label"),
        "risk_probability": row.get("risk_probability"),
        "risk_threshold": row.get("risk_threshold"),
        "lead_time_bucket": row.get("lead_time_bucket"),
        "lead_time_confidence": row.get("lead_time_confidence"),
        "top_sensors": json.dumps(top_sensors, ensure_ascii=False),
        "sensor_scores": json.dumps(sensor_scores, ensure_ascii=False),
        "evidence_details": json.dumps(evidences, ensure_ascii=False),
        "diagnostic_label": "holdout_overestimated_risk",
        "pattern_notes": "정상 holdout window가 risk threshold를 넘어 priority 입력에서 과대평가될 수 있음",
        "explainability_run_id": RUN_ID,
        "source_model_run_id": metadata.get("run_id"),
    })

false_positive_window_evidence = pd.DataFrame(fp_rows)
decision_feature_evidence = window_evidence.copy()
decision_feature_evidence["priority_input_role"] = "risk_or_anomaly_evidence"
if not false_positive_window_evidence.empty:
    overestimated = false_positive_window_evidence.copy()
    overestimated["priority_input_role"] = "overestimated_risk_diagnostic"
    decision_feature_evidence = pd.concat([decision_feature_evidence, overestimated], ignore_index=True, sort=False)

fp_feature_rows = []
for _, fp_row in false_positive_window_evidence.iterrows():
    for item in json.loads(fp_row["evidence_details"]):
        fp_feature_rows.append({
            "feature": item["feature"],
            "family": item["family"],
            "direction": item["direction"],
            "evidence_score": item["evidence_score"],
            "abs_robust_z": abs(item["robust_z"]),
        })

if fp_feature_rows:
    false_positive_feature_summary = pd.DataFrame(fp_feature_rows).groupby(
        ["feature", "family", "direction"], as_index=False
    ).agg(
        count=("feature", "size"),
        mean_evidence_score=("evidence_score", "mean"),
        max_evidence_score=("evidence_score", "max"),
        mean_abs_robust_z=("abs_robust_z", "mean"),
    ).sort_values(["count", "mean_evidence_score"], ascending=[False, False]).reset_index(drop=True)
else:
    false_positive_feature_summary = pd.DataFrame(columns=["feature", "family", "direction", "count", "mean_evidence_score", "max_evidence_score", "mean_abs_robust_z"])

display(false_positive_window_evidence.head(20))
display(false_positive_feature_summary.head(20))

## 5. 과대위험 group 진단

06에서 확인한 group별 과대위험 판단 문제를 07에서 우선 설명 대상으로 정리한다.

In [ ]:
false_positive_groups = group_diagnostics.copy()
false_positive_groups = false_positive_groups[
    false_positive_groups["split"].eq("holdout") & (false_positive_groups["false_positive_rate"] > 0)
].sort_values(["false_positive_rate", "false_positive"], ascending=[False, False]).reset_index(drop=True)

def make_group_note(row: pd.Series) -> str:
    return (
        f"holdout {row['group_column']}={row['group_value']} 그룹에서 "
        f"정상 window 과대위험 비율 {row['false_positive_rate']:.3f}, "
        f"과대위험 {int(row['false_positive'])}건이 발생했다. "
        "해당 그룹은 08 decision_features 또는 후속 Priority Engine에서 별도 보정 후보로 본다."
    )

false_positive_groups["diagnostic_note"] = false_positive_groups.apply(make_group_note, axis=1)
display(false_positive_groups.head(20))

## 6. 저장

07 결과는 최신본과 run별 파일을 함께 저장한다.

In [ ]:
def save_latest_and_run(df_out: pd.DataFrame, latest_name: str):
    latest_path = OUTPUT_DIR / latest_name
    run_path = RUN_DIR / latest_name
    df_out.to_csv(latest_path, index=False)
    df_out.to_csv(run_path, index=False)
    return latest_path, run_path

save_latest_and_run(importance_summary, "feature_importance_summary.csv")
save_latest_and_run(top_importance, "top_feature_importance.csv")
save_latest_and_run(family_importance, "feature_family_importance.csv")
save_latest_and_run(window_evidence, "window_evidence.csv")
save_latest_and_run(decision_feature_evidence, "decision_feature_evidence.csv")
save_latest_and_run(false_positive_window_evidence, "false_positive_window_evidence.csv")
save_latest_and_run(false_positive_feature_summary, "false_positive_feature_summary.csv")
save_latest_and_run(false_positive_groups, "false_positive_group_diagnostics.csv")

explainability_metadata = {
    "run_id": RUN_ID,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "source_model_run_id": metadata.get("run_id"),
    "source_risk_model_version": metadata.get("risk_model_version"),
    "source_leadtime_model_version": metadata.get("leadtime_model_version"),
    "method": "LightGBM feature importance + robust deviation evidence baseline",
    "notes": [
        "SHAP 기반 국소 설명이 아니라 feature importance와 normal reference 대비 벗어난 정도를 결합한 baseline 근거다.",
        "07 결과는 08 decision_features/export의 top_sensors, sensor_scores, pattern_notes 후보로 사용한다.",
    ],
    "output_files": [
        "feature_importance_summary.csv",
        "top_feature_importance.csv",
        "feature_family_importance.csv",
        "window_evidence.csv",
        "decision_feature_evidence.csv",
        "false_positive_window_evidence.csv",
        "false_positive_feature_summary.csv",
        "false_positive_group_diagnostics.csv",
    ],
}

for path in [OUTPUT_DIR / "explainability_metadata.json", RUN_DIR / "explainability_metadata.json"]:
    path.write_text(json.dumps(explainability_metadata, ensure_ascii=False, indent=2), encoding="utf-8")

print("saved to:", OUTPUT_DIR)
print("run dir:", RUN_DIR)